# 07 · Implement-from-scratch drills

Eighteen "implement X in 10 minutes" questions of the kind AI-engineer interviews use as warm-ups or whole rounds. Each one gets the ask, the one idea that makes it work, a short implementation, and **assert tests**. Where a library does the same job (NumPy, SciPy, PyTorch, scikit-learn, `rank_bm25`), the tests check against it. Cover the code, write it yourself, then run the tests.

**Time:** ~60 min (about 3 min per drill) · **Runs:** offline on CPU in < 1 min · **Needs:** [NumPy for AI engineers](../04_numpy_pandas_pytorch/01_numpy_for_ai_engineers.ipynb)

## Why this matters in interviews

- Live-coding a numerically stable softmax, attention with a causal mask, or BM25 is the fastest way for an interviewer to check that you *understand* the thing you put on your CV.
- The engineering drills (LRU cache, token bucket, retry with backoff, chunking, RRF) are the building blocks of every LLM service. They also show up as "small design" questions.
- The grading is not just "does it run". They want the edge cases (overflow, empty input, ties) and a test you wrote yourself.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `fo22` | What are logits, and what is the softmax doing? |
| `tr03` | What is self-attention, in your own words? |
| `tr06` | What is the causal mask and why is it essential? |
| `tr09` | What does LayerNorm do, and how is RMSNorm different? |
| `fo03` | Why do LLMs use a subword tokenizer instead of treating every word as one token? |
| `fo15` | What are top-k and top-p, and which should you use? |
| `ve01` | What is cosine similarity, and what are the alternatives? |
| `ve08` | What is BM25 and how does it work? |
| `rg45` | What is reciprocal rank fusion and why is it the default? |
| `po08` | What is exponential backoff, and why is jitter not optional? |

## Table of contents

| # | Drill | Tested against |
|---|---|---|
| 1 | Numerically stable softmax + cross-entropy | `scipy.special.softmax`, `torch.nn.functional.cross_entropy` |
| 2 | Cosine-similarity top-k | `sklearn.metrics.pairwise.cosine_similarity` |
| 3 | Scaled dot-product attention (+ causal mask) | `torch.nn.functional.scaled_dot_product_attention` |
| 4 | Layer norm (and RMSNorm) | `torch.nn.functional.layer_norm` |
| 5 | One BPE merge step | the textbook `low/lower/newest/widest` example |
| 6 | k-means | `sklearn.cluster.KMeans` |
| 7 | Logistic regression with gradient descent | `sklearn.linear_model.LogisticRegression` |
| 8 | ROUGE-1 | a hand-computed example |
| 9 | Edit (Levenshtein) distance | a brute-force recursive version |
| 10 | LRU cache | eviction-order tests |
| 11 | Token-bucket rate limiter | a fake clock |
| 12 | Retry-with-backoff decorator | a flaky function and a fake `sleep` |
| 13 | Chunk text with overlap | coverage and overlap invariants |
| 14 | Reciprocal rank fusion | a hand-worked ranking |
| 15 | Top-p (nucleus) sampling | the definition: smallest set with mass ≥ p |
| 16 | Beam search on a toy model | brute-force search over all sequences |
| 17 | BM25 scoring | `rank_bm25.BM25Okapi` |
| 18 | Greedy vs beam, and length normalisation | a two-line extension of drill 16 |

In [ ]:
import functools
import math
import random
import time
from collections import Counter, OrderedDict, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

rng = np.random.default_rng(0)
torch.manual_seed(0)
random.seed(0)
print("numpy", np.__version__, "| torch", torch.__version__)

### 1 · Numerically stable softmax and cross-entropy

**Ask:** *"Implement softmax. Now make it not overflow. Now cross-entropy from logits."* **The idea:** softmax is shift-invariant, so subtract the row max before `exp`. The largest exponent is then `e^0 = 1`. For cross-entropy, use **log-softmax** (`z - logsumexp(z)`) and never take the `log` of a softmax output, which can be exactly 0.

In [ ]:
from scipy.special import softmax as scipy_softmax

def softmax(x, axis=-1):
    x = np.asarray(x, dtype=np.float64)
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def cross_entropy(logits, targets):
    z = np.asarray(logits, dtype=np.float64)
    z = z - z.max(axis=1, keepdims=True)
    log_probs = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
    return -log_probs[np.arange(len(targets)), targets].mean()

logits = rng.normal(size=(4, 6)) * 3
with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp([1000.0, 1001.0]) / np.exp([1000.0, 1001.0]).sum()
print("naive softmax([1000, 1001]) =", naive, "| stable =", softmax([1000.0, 1001.0]).round(4))
assert np.allclose(softmax(logits), scipy_softmax(logits, axis=-1))
assert np.allclose(softmax(logits).sum(axis=1), 1.0) and np.isfinite(softmax([1000.0, 1001.0])).all()
targets = np.array([0, 3, 5, 1])
ours, theirs = cross_entropy(logits, targets), F.cross_entropy(torch.tensor(logits), torch.tensor(targets)).item()
print(f"cross-entropy ours {ours:.6f} vs torch {theirs:.6f}")
assert abs(ours - theirs) < 1e-9

### 2 · Cosine-similarity top-k

**Ask:** *"Given a query vector and a matrix of document vectors, return the k most similar."* **The idea:** normalise, then a single matrix-vector product. `np.argpartition` finds the top k in O(n), and you sort only those k. Say it: "at millions of rows this is exactly what a vector DB does, with an ANN index instead of a full scan."

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def cosine_top_k(query, matrix, k):
    q = query / np.linalg.norm(query)
    m = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    sims = m @ q
    k = min(k, len(sims))
    idx = np.argpartition(-sims, k - 1)[:k]
    idx = idx[np.argsort(-sims[idx], kind="stable")]
    return idx, sims[idx]

docs, query = rng.normal(size=(1000, 64)), rng.normal(size=64)
idx, sims = cosine_top_k(query, docs, 5)
ref = cosine_similarity(query[None], docs)[0]
print("top-5 ids:", idx, "sims:", sims.round(3))
assert list(idx) == list(np.argsort(-ref)[:5]) and np.allclose(sims, np.sort(ref)[::-1][:5])
assert len(cosine_top_k(query, docs[:3], 10)[0]) == 3            # k larger than n

### 3 · Scaled dot-product attention, with a causal mask

**Ask:** *"Write attention. Why divide by √d? Add a causal mask."* **The idea:** `softmax(QKᵀ/√d) V`. Dividing by √d keeps the score variance near 1, so the softmax does not saturate. The causal mask sets scores for *future* positions to −∞ **before** the softmax, so each token attends only to itself and the past.

In [ ]:
def attention(Q, K, V, causal=False):
    d = Q.shape[-1]
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d)
    if causal:
        T_q, T_k = scores.shape[-2:]
        future = np.triu(np.ones((T_q, T_k), dtype=bool), k=1)
        scores = np.where(future, -np.inf, scores)
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

B, T, D = 2, 6, 16
Q, K, V = (rng.normal(size=(B, T, D)) for _ in range(3))
for causal in (False, True):
    out, w = attention(Q, K, V, causal=causal)
    ref = F.scaled_dot_product_attention(*(torch.tensor(a) for a in (Q, K, V)), is_causal=causal).numpy()
    assert np.allclose(out, ref, atol=1e-10), causal
_, w = attention(Q, K, V, causal=True)
assert np.allclose(np.triu(w[0], k=1), 0) and np.allclose(w.sum(-1), 1)
print("matches torch.scaled_dot_product_attention (causal and not); future weights are exactly 0")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, causal in zip(axes, (False, True)):
    im = ax.imshow(attention(Q, K, V, causal=causal)[1][0], cmap="Blues", vmin=0, vmax=1)
    ax.set_title("causal mask" if causal else "no mask"); ax.set_xlabel("key position (attended to)")
    ax.set_ylabel("query position")
fig.colorbar(im, ax=axes, shrink=0.8, label="attention weight"); plt.show()

### 4 · Layer norm (and RMSNorm)

**Ask:** *"Implement LayerNorm. How is RMSNorm different?"* **The idea:** normalise each token's feature vector to zero mean and unit variance, then apply a learned scale `γ` and shift `β`. Use the **biased** variance (divide by n) plus `eps`. RMSNorm skips the mean subtraction and `β`, dividing by the root-mean-square. That is cheaper, and it is what Llama-style models use.

In [ ]:
def layer_norm(x, gamma, beta, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)                      # biased, like PyTorch
    return gamma * (x - mu) / np.sqrt(var + eps) + beta

def rms_norm(x, gamma, eps=1e-6):
    return gamma * x / np.sqrt((x ** 2).mean(axis=-1, keepdims=True) + eps)

x = rng.normal(3.0, 2.0, size=(4, 10, 32))
gamma, beta = rng.normal(size=32), rng.normal(size=32)
ref = F.layer_norm(torch.tensor(x), (32,), torch.tensor(gamma), torch.tensor(beta), eps=1e-5).numpy()
assert np.allclose(layer_norm(x, gamma, beta), ref, atol=1e-10)
y = layer_norm(x, np.ones(32), np.zeros(32))
print(f"after LN: mean {y.mean(-1).std():.1e} ~ 0, std {y.std(-1).mean():.4f} ~ 1 | RMSNorm rms "
      f"{np.sqrt((rms_norm(x, np.ones(32)) ** 2).mean(-1)).mean():.4f}")
assert np.allclose(y.mean(-1), 0, atol=1e-9) and np.allclose(y.std(-1), 1, atol=1e-3)

### 5 · One BPE merge step

**Ask:** *"Given word counts, perform one step of byte-pair encoding training."* **The idea:** count every adjacent symbol pair, weighted by word frequency. Then merge the most frequent pair everywhere it occurs. Repeat to build the vocabulary. Common pieces ("est") become single tokens, and rare words still split into known pieces, which is why LLMs use subwords (`fo03`).

In [ ]:
def bpe_merge_step(word_counts):
    """word_counts: {tuple_of_symbols: count}. Returns (best_pair, new_word_counts)."""
    pairs = Counter()
    for word, count in word_counts.items():
        for a, b in zip(word, word[1:]):
            pairs[a, b] += count
    best = min(pairs, key=lambda p: (-pairs[p], p))           # most frequent; ties -> alphabetical
    merged = {}
    for word, count in word_counts.items():
        out, i = [], 0
        while i < len(word):
            if i + 1 < len(word) and (word[i], word[i + 1]) == best:
                out.append(word[i] + word[i + 1]); i += 2
            else:
                out.append(word[i]); i += 1
        merged[tuple(out)] = merged.get(tuple(out), 0) + count
    return best, merged

vocab = {tuple("low") + ("</w>",): 5, tuple("lower") + ("</w>",): 2,
         tuple("newest") + ("</w>",): 6, tuple("widest") + ("</w>",): 3}
merges = []
for _ in range(3):
    pair, vocab = bpe_merge_step(vocab)
    merges.append(pair)
    print(f"merge {pair} ->", [" ".join(w) for w in vocab])
assert merges[:2] == [("e", "s"), ("es", "t")]
assert ("n", "e", "w", "est", "</w>") in vocab or ("n", "e", "w", "est</w>") in vocab

### 6 · k-means

**Ask:** *"Implement k-means."* **The idea:** alternate two steps until nothing changes. First assign each point to its nearest centroid, then move each centroid to the mean of its points. Say the caveats: the result depends on the initialisation (use several restarts or k-means++), an empty cluster needs handling, and you have to choose k (elbow or silhouette).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score

def kmeans(X, k, n_iter=100, seed=0, restarts=5):
    best = None
    for r in range(restarts):
        g = np.random.default_rng(seed + r)
        C = X[g.choice(len(X), k, replace=False)]
        for _ in range(n_iter):
            labels = ((X[:, None, :] - C[None]) ** 2).sum(-1).argmin(1)
            new_C = np.array([X[labels == j].mean(0) if (labels == j).any() else C[j] for j in range(k)])
            if np.allclose(new_C, C):
                break
            C = new_C
        inertia = ((X - C[labels]) ** 2).sum()
        if best is None or inertia < best[2]:
            best = (C, labels, inertia)
    return best

X, y_true = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=0)
C, labels, inertia = kmeans(X, 3)
sk = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
agree = adjusted_rand_score(sk.labels_, labels)
print(f"inertia ours {inertia:.1f} vs sklearn {sk.inertia_:.1f} | agreement with sklearn's clusters (ARI) {agree:.3f} "
      f"| ARI vs the true blobs {adjusted_rand_score(y_true, labels):.3f} (blobs overlap, so < 1 for sklearn too)")
assert inertia <= sk.inertia_ * 1.01 and agree > 0.99

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(X[:, 0], X[:, 1], c=labels, cmap="viridis", s=12)
ax.scatter(C[:, 0], C[:, 1], c="red", marker="X", s=200, label="centroids")
ax.set_title("k-means from scratch (k = 3)"); ax.set_xlabel("x1"); ax.set_ylabel("x2"); ax.legend()
plt.tight_layout(); plt.show()

### 7 · Logistic regression with gradient descent

**Ask:** *"Train logistic regression with plain gradient descent. Derive the gradient."* **The idea:** `p = σ(Xw + b)`. The gradient of the mean log-loss is simply `Xᵀ(p − y)/n`, which is the "prediction minus label" pattern. Standardise the features, or gradient descent crawls. Clip the logits (or use `np.logaddexp`) so the loss never evaluates `log(0)`.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def train_logreg(X, y, lr=0.5, epochs=500, l2=1e-3):
    w, b, losses = np.zeros(X.shape[1]), 0.0, []
    for _ in range(epochs):
        z = X @ w + b
        p = 1 / (1 + np.exp(-z))
        losses.append(np.mean(np.logaddexp(0, z) - y * z) + 0.5 * l2 * w @ w)   # stable log-loss
        w -= lr * (X.T @ (p - y) / len(y) + l2 * w)
        b -= lr * (p - y).mean()
    return w, b, losses

Xb, yb = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.25, stratify=yb, random_state=0)
scaler = StandardScaler().fit(Xtr)
Xtr, Xte = scaler.transform(Xtr), scaler.transform(Xte)
w, b, losses = train_logreg(Xtr, ytr)
acc_ours = ((Xte @ w + b > 0) == yte).mean()
acc_sk = LogisticRegression(max_iter=1000).fit(Xtr, ytr).score(Xte, yte)
print(f"test accuracy ours {acc_ours:.3f} vs sklearn {acc_sk:.3f}; loss {losses[0]:.3f} -> {losses[-1]:.3f}")
assert abs(acc_ours - acc_sk) <= 0.03 and losses[-1] < losses[0] and all(np.diff(losses) <= 1e-12)

### 8 · ROUGE-1

**Ask:** *"Implement ROUGE-1 precision, recall and F1."* **The idea:** clipped unigram overlap. Count the words both texts share, with each word counted at most as often as it appears in either (`Counter & Counter`). Recall is overlap / reference length, precision is overlap / candidate length. ROUGE is recall-oriented, which fits summaries.

In [ ]:
def rouge1(candidate, reference):
    c, r = Counter(candidate.lower().split()), Counter(reference.lower().split())
    overlap = sum((c & r).values())
    if overlap == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    p, rec = overlap / sum(c.values()), overlap / sum(r.values())
    return {"precision": p, "recall": rec, "f1": 2 * p * rec / (p + rec)}

s = rouge1("the cat sat on the mat", "the cat is on the mat")
print({k: round(v, 3) for k, v in s.items()})
assert s["precision"] == 5 / 6 and s["recall"] == 5 / 6              # the,cat,on,the,mat
assert rouge1("the the the", "the cat")["precision"] == 1 / 3        # clipped: 'the' counts once
assert rouge1("", "anything")["f1"] == 0.0

### 9 · Edit distance

**Ask:** *"Minimum number of insertions, deletions and substitutions to turn `a` into `b`."* **The idea:** dynamic programming, `D[i][j] = min(D[i-1][j] + 1, D[i][j-1] + 1, D[i-1][j-1] + (a[i] != b[j]))`. Keep only two rows, which gives O(len(b)) memory. It is used for fuzzy matching, OCR clean-up and WER in speech.

In [ ]:
def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

@functools.lru_cache(maxsize=None)
def edit_brute(a, b):
    if not a or not b:
        return len(a) + len(b)
    return min(edit_brute(a[1:], b) + 1, edit_brute(a, b[1:]) + 1, edit_brute(a[1:], b[1:]) + (a[0] != b[0]))

assert edit_distance("kitten", "sitting") == 3 and edit_distance("", "abc") == 3 and edit_distance("flaw", "lawn") == 2
pairs = ["".join(random.choices("abc", k=random.randint(0, 7))) for _ in range(80)]
assert all(edit_distance(x, y) == edit_brute(x, y) for x, y in zip(pairs, pairs[1:]))
print("kitten -> sitting:", edit_distance("kitten", "sitting"), "| 79 random pairs match the brute-force recursion")

### 10 · LRU cache

**Ask:** *"Implement an LRU cache with O(1) `get` and `put`."* **The idea:** a hash map plus a doubly linked list in recency order. In Python, `OrderedDict` is exactly that: `move_to_end` on every access, and `popitem(last=False)` evicts the least recently used. If they forbid `OrderedDict`, write the dict + linked list. The logic is the same.

In [ ]:
class LRUCache:
    def __init__(self, capacity):
        self.capacity, self.data = capacity, OrderedDict()
    def get(self, key, default=None):
        if key not in self.data:
            return default
        self.data.move_to_end(key)                 # now most recently used
        return self.data[key]
    def put(self, key, value):
        if key in self.data:
            self.data.move_to_end(key)
        self.data[key] = value
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)          # evict least recently used

cache = LRUCache(2)
cache.put("a", 1); cache.put("b", 2)
assert cache.get("a") == 1                         # a is now the most recent
cache.put("c", 3)                                  # evicts b, not a
assert cache.get("b") is None and cache.get("a") == 1 and cache.get("c") == 3
cache.put("a", 10); cache.put("d", 4)              # updating a refreshes it -> c is evicted
assert list(cache.data) == ["a", "d"] and cache.get("a") == 10
print("eviction order correct:", list(cache.data.items()))

### 11 · Token-bucket rate limiter

**Ask:** *"Rate-limit each user to 5 requests per second with bursts."* **The idea:** each key has a bucket of up to `capacity` tokens that refills continuously at `rate` per second, and a request spends one token. Refill lazily from the elapsed time, so no background timer is needed. Inject the clock so tests are deterministic. In production the state lives in Redis, updated atomically with a Lua script.

In [ ]:
class TokenBucket:
    def __init__(self, capacity, rate, clock=time.monotonic):
        self.capacity, self.rate, self.clock, self.state = capacity, rate, clock, {}
    def allow(self, key="default"):
        now = self.clock()
        tokens, last = self.state.get(key, (float(self.capacity), now))
        tokens = min(self.capacity, tokens + (now - last) * self.rate)
        allowed = tokens >= 1
        self.state[key] = (tokens - 1 if allowed else tokens, now)
        return allowed

t = [0.0]
tb = TokenBucket(capacity=5, rate=5, clock=lambda: t[0])
burst = [tb.allow("u") for _ in range(7)]
assert burst == [True] * 5 + [False, False]
t[0] += 0.2                                      # 0.2 s x 5/s = 1 token back
assert tb.allow("u") and not tb.allow("u")
t[0] += 10                                       # a long pause refills only up to capacity
assert sum(tb.allow("u") for _ in range(10)) == 5
assert tb.allow("other-user")
print("burst of 7 ->", burst, "| refill and per-key isolation OK")

### 12 · Retry with exponential backoff (and jitter) as a decorator

**Ask:** *"Write a decorator that retries a flaky call."* **The idea:** retry **only retryable** errors (timeouts, 429, 5xx, never a 400), wait `base × 2^attempt` capped at a maximum, and add **jitter**. Without jitter, every client that failed together retries together and hammers the server in synchronised waves (`po08`). Inject `sleep` for tests.

In [ ]:
def retry(max_attempts=4, base=0.2, max_delay=5.0, retry_on=(TimeoutError, ConnectionError),
          sleep=time.sleep, jitter=random.random):
    def decorate(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except retry_on:
                    if attempt == max_attempts:
                        raise
                    sleep(min(max_delay, base * 2 ** (attempt - 1)) * jitter())   # "full jitter"
        return wrapper
    return decorate

delays, calls = [], {"n": 0}
@retry(sleep=delays.append, jitter=lambda: 1.0)          # jitter off here, so the delays are exact
def flaky():
    calls["n"] += 1
    if calls["n"] < 3:
        raise TimeoutError("upstream slow")
    return "ok"

assert flaky() == "ok" and calls["n"] == 3 and delays == [0.2, 0.4]
@retry(sleep=delays.append)
def bad_request():
    raise ValueError("400: your fault")                    # not retryable -> raised at once
try:
    bad_request()
except ValueError:
    pass
@retry(max_attempts=3, sleep=lambda s: None)
def always_down():
    raise ConnectionError("down")
try:
    always_down(); raise AssertionError("should have raised")
except ConnectionError:
    pass
print("retried twice with delays", delays[:2], "| non-retryable raised at once | gives up after max_attempts")

### 13 · Chunk text with overlap

**Ask:** *"Split a document into chunks of at most N characters with overlap M."* **The idea:** step by `size − overlap`. Check the invariants: every character is covered, consecutive chunks share exactly `overlap` characters, there are no empty chunks, and `overlap < size` (otherwise the loop never advances). Mention that real splitters prefer paragraph, sentence or word boundaries ([the RAG skeleton](01_rag_skeleton.ipynb) uses `RecursiveCharacterTextSplitter`).

In [ ]:
def chunk_text(text, size=200, overlap=40):
    if not 0 <= overlap < size:
        raise ValueError("need 0 <= overlap < size")
    if not text:
        return []
    step = size - overlap
    return [text[i:i + size] for i in range(0, max(len(text) - overlap, 1), step)]

doc = " ".join(f"sentence {i} of the manual." for i in range(60))
chunks = chunk_text(doc, size=200, overlap=40)
assert all(0 < len(c) <= 200 for c in chunks)
assert all(a[-40:] == b[:40] for a, b in zip(chunks, chunks[1:]))            # exact overlap
rebuilt = chunks[0] + "".join(c[40:] for c in chunks[1:])
assert rebuilt == doc                                                          # nothing lost, nothing added
assert chunk_text("short", 200, 40) == ["short"] and chunk_text("", 10, 2) == []
print(f"{len(doc)} chars -> {len(chunks)} chunks of <= 200 with 40 overlap; rebuilt text is identical")

### 14 · Reciprocal rank fusion

**Ask:** *"You have a BM25 ranking and a vector ranking. Merge them."* **The idea:** `score(d) = Σ 1/(k + rank_i(d))` with `k = 60`. It uses **ranks, not scores**, because the two lanes' scores are on different scales. A document ranked well by *both* lanes beats one ranked first by only one.

In [ ]:
def rrf(rankings, k=60):
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc] += 1.0 / (k + rank)
    return sorted(scores, key=lambda d: (-scores[d], d)), dict(scores)

bm25_rank = ["doc_code", "doc_b", "doc_c", "doc_d"]
dense_rank = ["doc_para", "doc_b", "doc_e", "doc_code"]
fused, scores = rrf([bm25_rank, dense_rank])
print("fused:", fused[:4], {d: round(s, 5) for d, s in scores.items() if d in fused[:3]})
assert fused[0] == "doc_b"                      # 2nd in both beats 1st + 4th (1/62+1/62 > 1/61+1/64)
assert fused.index("doc_code") < fused.index("doc_para")

### 15 · Top-p (nucleus) sampling

**Ask:** *"Implement top-p sampling with temperature."* **The idea:** divide the logits by the temperature, softmax, sort descending, and keep the **smallest set of tokens whose cumulative probability reaches p**. Renormalise that set and sample from it. Top-p adapts to the model's confidence: a peaked distribution keeps one or two tokens, a flat one keeps many (`fo15`).

In [ ]:
def top_p_filter(probs, p):
    order = np.argsort(-probs, kind="stable")
    cum = np.cumsum(probs[order])
    n_keep = int(np.searchsorted(cum, p - 1e-12) + 1)       # first index where cum >= p
    keep = order[:n_keep]
    out = np.zeros_like(probs)
    out[keep] = probs[keep]
    return out / out.sum(), keep

def sample_top_p(logits, p=0.9, temperature=1.0, rng=rng):
    probs = softmax(np.asarray(logits) / temperature)
    filtered, _ = top_p_filter(probs, p)
    return int(rng.choice(len(filtered), p=filtered))

probs = np.array([0.40, 0.25, 0.15, 0.10, 0.05, 0.03, 0.02])
filtered, keep = top_p_filter(probs, 0.8)
assert set(keep) == {0, 1, 2} and probs[keep].sum() >= 0.8 and probs[keep[:-1]].sum() < 0.8   # minimal set
assert len(top_p_filter(probs, 1.0)[1]) == len(probs) and len(top_p_filter(probs, 0.01)[1]) == 1
draws = Counter(sample_top_p(np.log(probs), p=0.8) for _ in range(2000))
assert set(draws) <= {0, 1, 2}
print("nucleus for p=0.8:", sorted(keep.tolist()), "| 2000 samples only ever hit:", dict(sorted(draws.items())))

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(range(len(probs)), probs, color=["#4c6ef5" if i in keep else "#dee2e6" for i in range(len(probs))])
ax.set_xticks(range(len(probs)), [f"tok{i}" for i in range(len(probs))]); ax.set_ylabel("probability")
ax.set_title("Top-p = 0.8 keeps the blue tokens (cumulative mass 0.80); grey ones can never be sampled")
plt.tight_layout(); plt.show()

### 16 · Beam search on a toy model

**Ask:** *"Implement beam search."* **The idea:** keep the `B` best partial sequences by **summed log-probability**. Extend each one by every token, keep the best `B` again, and repeat. Greedy decoding is beam search with `B = 1`. It can miss a sequence whose *first* token is less likely but whose continuation is much more likely. The toy "model" below is a bigram table, small enough to check against brute force over all 4³ sequences.

In [ ]:
import itertools

VOCAB = ["A", "B", "C", "D"]
START = {"A": 0.5, "B": 0.4, "C": 0.05, "D": 0.05}
NEXT = {"A": {"A": 0.3, "B": 0.3, "C": 0.2, "D": 0.2},      # after A: uncertain
        "B": {"A": 0.05, "B": 0.05, "C": 0.85, "D": 0.05},  # after B: very sure
        "C": {"A": 0.1, "B": 0.1, "C": 0.1, "D": 0.7},
        "D": {"A": 0.25, "B": 0.25, "C": 0.25, "D": 0.25}}

def logp(seq):
    return math.log(START[seq[0]]) + sum(math.log(NEXT[a][b]) for a, b in zip(seq, seq[1:]))

def beam_search(length, beam_width):
    beams = [((t,), math.log(p)) for t, p in START.items()]
    beams = sorted(beams, key=lambda x: -x[1])[:beam_width]
    for _ in range(length - 1):
        candidates = [(seq + (t,), s + math.log(NEXT[seq[-1]][t])) for seq, s in beams for t in VOCAB]
        beams = sorted(candidates, key=lambda x: (-x[1], x[0]))[:beam_width]
    return beams[0]

best_brute = max(itertools.product(VOCAB, repeat=3), key=logp)
greedy_seq, greedy_lp = beam_search(3, beam_width=1)
beam_seq, beam_lp = beam_search(3, beam_width=3)
print(f"greedy  {''.join(greedy_seq)}  p={math.exp(greedy_lp):.4f}")
print(f"beam=3  {''.join(beam_seq)}  p={math.exp(beam_lp):.4f}   brute force best {''.join(best_brute)} p={math.exp(logp(best_brute)):.4f}")
assert beam_seq == best_brute and greedy_lp < beam_lp
assert beam_search(3, 64)[0] == best_brute                               # a wide enough beam = exhaustive

### 17 · BM25 scoring

**Ask:** *"Score documents for a query with BM25."* **The idea:** for each query term, IDF × saturated term frequency. `tf·(k1+1) / (tf + k1·(1 − b + b·len/avgdl))`, where `k1` (≈1.2–2) makes repeated terms count less and less, and `b` (0.75) normalises by document length. The implementation below reproduces `rank_bm25.BM25Okapi` exactly, including its quirk: terms that appear in more than half the documents get a negative raw IDF, which is floored to `0.25 × mean IDF`. Lucene avoids this with `log(1 + (N − n + 0.5)/(n + 0.5))`.

In [ ]:
from rank_bm25 import BM25Okapi

def bm25_scores(query_tokens, corpus_tokens, k1=1.5, b=0.75, epsilon=0.25):
    N = len(corpus_tokens)
    avgdl = sum(map(len, corpus_tokens)) / N
    df = Counter(t for doc in corpus_tokens for t in set(doc))
    idf = {t: math.log(N - n + 0.5) - math.log(n + 0.5) for t, n in df.items()}
    floor = epsilon * sum(idf.values()) / len(idf)
    idf = {t: (v if v >= 0 else floor) for t, v in idf.items()}
    scores = []
    for doc in corpus_tokens:
        tf, s = Counter(doc), 0.0
        for t in query_tokens:
            if t in tf:
                s += idf[t] * tf[t] * (k1 + 1) / (tf[t] + k1 * (1 - b + b * len(doc) / avgdl))
        scores.append(s)
    return np.array(scores)

corpus = ["error e-5003 means the warehouse api timed out",
          "refunds are paid within five business days",
          "the api returns error codes for failed payments",
          "express shipping costs fifteen dollars",
          "the warranty covers manufacturing defects"]
tokens = [d.split() for d in corpus]
query = "error e-5003".split()
ours, theirs = bm25_scores(query, tokens), BM25Okapi(tokens).get_scores(query)
print("ours  :", ours.round(4), "\nrank_bm25:", np.asarray(theirs).round(4))
assert np.allclose(ours, theirs) and ours.argmax() == 0                 # the exact-code doc ranks first

### 18 · Greedy vs beam, and length normalisation

**Ask (follow-up to 16):** *"Why do beam searches prefer short outputs, and what is the fix?"* **The idea:** every extra token adds a negative log-probability, so longer hypotheses always score lower. When hypotheses can end at different lengths (at an EOS token), rank them by `score / length^α` with α ≈ 0.6–1.0. Here is the effect on two hypotheses:

In [ ]:
def normalized(logprob, length, alpha=0.7):
    return logprob / (length ** alpha)

short = (math.log(0.6) + math.log(0.5), 2)                     # two decent tokens, then EOS
long = (math.log(0.4) + 4 * math.log(0.8), 5)                 # five confident tokens
print(f"raw log-prob:      short {short[0]:.3f}  long {long[0]:.3f}  -> raw picks {'short' if short[0] > long[0] else 'long'}")
print(f"normalised (a=.7): short {normalized(*short):.3f}  long {normalized(*long):.3f}  "
      f"-> normalised picks {'short' if normalized(*short) > normalized(*long) else 'long'}")
assert short[0] > long[0] and normalized(*long) > normalized(*short)

## Try it yourself

**1.** Implement **top-k filtering** (keep the k highest-probability tokens, renormalise). Test that it keeps exactly k tokens, that its output sums to 1, and that `k = 1` is greedy.

In [ ]:
# Solution — try it yourself first, then run this
def top_k_filter(probs, k):
    keep = np.argsort(-probs, kind="stable")[:k]
    out = np.zeros_like(probs)
    out[keep] = probs[keep]
    return out / out.sum()

f3 = top_k_filter(probs, 3)
assert (f3 > 0).sum() == 3 and np.isclose(f3.sum(), 1) and top_k_filter(probs, 1).argmax() == probs.argmax()
print("top-3:", f3.round(3))

**2.** Implement a **sliding-window log** rate limiter (at most N requests in any rolling window of W seconds) and show one case where it and the token bucket disagree.

In [ ]:
# Solution — try it yourself first, then run this
from collections import deque

class SlidingWindowLimiter:
    def __init__(self, limit, window, clock):
        self.limit, self.window, self.clock, self.log = limit, window, clock, defaultdict(deque)
    def allow(self, key="default"):
        now, q = self.clock(), self.log[key]
        while q and q[0] <= now - self.window:
            q.popleft()
        if len(q) < self.limit:
            q.append(now)
            return True
        return False

t = [0.0]
sw = SlidingWindowLimiter(limit=5, window=1.0, clock=lambda: t[0])
tb = TokenBucket(capacity=5, rate=5, clock=lambda: t[0])
first = [(sw.allow(), tb.allow()) for _ in range(5)]          # both allow a burst of 5 at t=0
t[0] = 0.5
second = (sw.allow(), tb.allow())                             # bucket refilled 2.5 tokens; window still full
print("t=0 burst:", first, "| t=0.5 -> sliding window, token bucket:", second)
assert second == (False, True)

## Say it in an interview

- **Talk while you type:** state the idea in one sentence ("subtract the max, because softmax is shift-invariant"), write the happy path, then the edge cases, then a test.
- **Edge cases they look for:** overflow (softmax, log-loss), empty inputs (chunking, ROUGE), `k > n` (top-k), ties (BPE, RRF, sorting), `overlap ≥ size` (infinite loop), non-retryable errors (retry), and injected clocks and sleeps (rate limiter, backoff).
- **Complexities to quote:** attention is O(T²·d) and the causal mask does not reduce it. Top-k with `argpartition` is O(n). An LRU cache is O(1) per operation. Edit distance is O(nm) time and O(m) memory. k-means is O(n·k·d) per iteration. BM25 is O(query terms × documents) without an inverted index.
- **Say which library you would use in real code** (PyTorch's SDPA, `rank_bm25` or OpenSearch, `tenacity` for retries, Redis for rate limits), and that the from-scratch version is how you prove you understand it.

## Next

- Foundations behind the drills: [attention from scratch](../07_genai_foundations/03_attention_from_scratch.ipynb) · [tokenization and BPE](../07_genai_foundations/01_tokenization_bpe_and_tiktoken.ipynb) · [decoding: temperature, top-k, top-p](../07_genai_foundations/05_decoding_temperature_top_k_top_p.ipynb) · [BM25 from scratch](../09_rag/02_bm25_from_scratch.ipynb) · [retries, backoff, rate limits](../16_production/02_retries_backoff_rate_limits_circuit_breakers.ipynb) · [tries, union-find, LRU cache](../03_dsa/14_tries_union_find_lru_cache.ipynb) · [dynamic programming](../03_dsa/12_dynamic_programming.ipynb)
- Other rounds: [RAG skeleton](01_rag_skeleton.ipynb) · [ML model end to end](06_ml_model_end_to_end.ipynb)
- Theory: [dl_fundamentals course](../../dl_fundamentals/index.html) · [dsa_basics course](../../dsa_basics/index.html) · [interview bank](../../ai_interview_prep/index.html)